# DDInter → PharmPilot Interaction Bundle (self-contained)
Builds a `bundle-v1` SQLite from DDInter 2.0 and downloads it — **no repo clone or login needed**. The normalize() + builder logic below is vendored verbatim from the PharmPilot engine so the bundle's drug tokens match the engine's. Then install the file in **Admin → Interaction Bundle**.

Runtime → Run all.

## 1. Vendored engine `normalize()` (salt-stripping + mineral-cation guard)

In [ ]:
"""Deterministic medication normalization for the CDS vertical slice."""


import re


BRAND_TO_GENERIC: dict[str, str] = {
    "biaxin": "clarithromycin",
    "zocor": "simvastatin",
    "prinivil": "lisinopril",
    "zestril": "lisinopril",
    "vasotec": "enalapril",
    "altace": "ramipril",
    "aldactone": "spironolactone",
    "inspra": "eplerenone",
    "glucophage": "metformin",
    "coumadin": "warfarin",
    "jantoven": "warfarin",
    "eliquis": "apixaban",
    "xarelto": "rivaroxaban",
    "pradaxa": "dabigatran",
    "lovenox": "enoxaparin",
    "advil": "ibuprofen",
    "motrin": "ibuprofen",
    "aleve": "naproxen",
    "celebrex": "celecoxib",
    "valium": "diazepam",
    "xanax": "alprazolam",
    "ativan": "lorazepam",
    "klonopin": "clonazepam",
    "lexapro": "escitalopram",
    "zoloft": "sertraline",
    "prozac": "fluoxetine",
    "paxil": "paroxetine",
    "celexa": "citalopram",
    "ultram": "tramadol",
}

GENERIC_CLASSES: dict[str, set[str]] = {
    "simvastatin": {"statin"},
    "atorvastatin": {"statin"},
    "lovastatin": {"statin"},
    "rosuvastatin": {"statin"},
    "pravastatin": {"statin"},
    "ibuprofen": {"nsaid"},
    "naproxen": {"nsaid"},
    "diclofenac": {"nsaid"},
    "meloxicam": {"nsaid"},
    "celecoxib": {"nsaid"},
    "warfarin": {"anticoagulant"},
    "apixaban": {"anticoagulant"},
    "rivaroxaban": {"anticoagulant"},
    "dabigatran": {"anticoagulant"},
    "edoxaban": {"anticoagulant"},
    "enoxaparin": {"anticoagulant"},
    "heparin": {"anticoagulant"},
    "alprazolam": {"benzodiazepine"},
    "clonazepam": {"benzodiazepine"},
    "diazepam": {"benzodiazepine"},
    "lorazepam": {"benzodiazepine"},
    "temazepam": {"benzodiazepine"},
    "oxazepam": {"benzodiazepine"},
    "sertraline": {"ssri"},
    "fluoxetine": {"ssri"},
    "paroxetine": {"ssri"},
    "citalopram": {"ssri"},
    "escitalopram": {"ssri"},
    "fluvoxamine": {"ssri"},
    "lisinopril": {"ace_inhibitor"},
    "enalapril": {"ace_inhibitor"},
    "ramipril": {"ace_inhibitor"},
    "benazepril": {"ace_inhibitor"},
    "captopril": {"ace_inhibitor"},
    "quinapril": {"ace_inhibitor"},
    "spironolactone": {"potassium_sparing_diuretic"},
    "eplerenone": {"potassium_sparing_diuretic"},
    "clarithromycin": {"macrolide_cyp3a4_inhibitor"},
    "erythromycin": {"macrolide_cyp3a4_inhibitor"},
    "metformin": {"biguanide"},
    "tramadol": {"serotonergic_opioid"},
    "amoxicillin": {"penicillin"},
    "ampicillin": {"penicillin"},
}


# Mineral cations whose salt/anion is part of the clinical identity — never strip when first.
_MINERAL_CATIONS = {
    "calcium", "magnesium", "sodium", "potassium", "iron", "ferrous", "ferric",
    "zinc", "aluminum", "aluminium",
}
# Counter-ions / esters / hydrates that are not the active when trailing an organic drug.
_SALT_TOKENS = {
    "hcl", "hydrochloride", "hbr", "hydrobromide", "sulfate", "sulphate", "bisulfate",
    "mesylate", "maleate", "tartrate", "besylate", "succinate", "fumarate", "tosylate",
    "citrate", "acetate", "carbonate", "phosphate", "gluconate", "bromide", "chloride",
    "dihydrate", "monohydrate", "hemihydrate", "anhydrous",
}


def normalize(name: str | None) -> str:
    if not name:
        return ""
    cleaned = re.sub(r"[^a-zA-Z0-9\s-]", " ", name).lower()
    cleaned = re.sub(r"\b(tablet|tab|capsule|cap|oral|solution|mg|mcg|ml|er|xr|sr)\b", " ", cleaned)
    # Drop strength/dose tokens (e.g. "25mg", "10", "0.5") — a drug name never
    # starts with a digit, but a strength suffix does. Without this, an
    # unrecognized drug like "diphenhydramine 25mg" normalizes to
    # "diphenhydramine 25mg" instead of "diphenhydramine", so downstream
    # exact-name lookups (e.g. the anticholinergic-burden table) miss.
    tokens = [token for token in re.split(r"[\s/-]+", cleaned) if token and not token[0].isdigit()]
    # Strip salt/counter-ion tokens for organic actives so "tizanidine hcl" → "tizanidine".
    # Guard: when the FIRST token is a mineral cation, the anion defines the product
    # (calcium citrate ≠ calcium carbonate), so keep the whole name.
    if tokens and tokens[0] not in _MINERAL_CATIONS:
        tokens = [tokens[0]] + [t for t in tokens[1:] if t not in _SALT_TOKENS]
    for token in tokens:
        if token in BRAND_TO_GENERIC:
            return BRAND_TO_GENERIC[token]
        if token in GENERIC_CLASSES:
            return token
    compact = " ".join(tokens)
    return BRAND_TO_GENERIC.get(compact, compact)


def classes_of(name: str | None) -> set[str]:
    return set(GENERIC_CLASSES.get(normalize(name), set()))
print('normalize() ready:', normalize('Tizanidine HCl 4mg'))


## 2. Vendored bundle builder (`RawInteraction`, `build_rules`, `write_bundle`)

In [ ]:
"""Build a bundle-v1 interaction SQLite from DDInter rows.

Format-agnostic: the Colab notebook adapts DDInter's CSVs into RawInteraction
rows and calls build_rules() + write_bundle(). Tokens are produced by the
engine's own normalize(), so a bundle rule matches the same real-Rx names the
engine derives — see tests/unit/test_ddinter_builder.py::test_end_to_end_alignment.
"""

import hashlib
import json
import sqlite3
from collections.abc import Iterable
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

SCHEMA_VERSION = "bundle-v1"  # vendored from interaction.bundle

_LEVEL_MAP = {"major": "Major", "moderate": "Moderate", "minor": "Minor",
              "contraindicated": "Contraindicated"}
_RANK = {"Minor": 0, "Moderate": 1, "Major": 2, "Contraindicated": 3}
_IMPORT_CONFIDENCE = 0.7


@dataclass
class RawInteraction:
    drug_a: str
    drug_b: str
    level: str
    mechanism: str = ""


def build_rules(rows: Iterable[RawInteraction]) -> list[dict]:
    best: dict[frozenset, dict] = {}
    for row in rows:
        a, b = normalize(row.drug_a), normalize(row.drug_b)
        if not a or not b or a == b:
            continue
        severity = _LEVEL_MAP.get((row.level or "").strip().lower(), "Minor")
        rule = {
            "kind": "drug_drug", "left": a, "right": b, "severity": severity,
            "mechanism": (row.mechanism or "").strip() or "DDInter-listed interaction.",
            "evidence": ["DDInter 2.0"], "confidence": _IMPORT_CONFIDENCE,
        }
        key = frozenset((a, b))
        if key not in best or _RANK[severity] > _RANK[best[key]["severity"]]:
            best[key] = rule
    return list(best.values())


def write_bundle(rules: list[dict], dest: Path, datasets: dict) -> dict:
    dest = Path(dest)
    if dest.exists():
        dest.unlink()
    payloads = sorted(json.dumps(r, sort_keys=True) for r in rules)
    checksum = hashlib.sha256("\n".join(payloads).encode()).hexdigest()
    meta = {
        "schema_version": SCHEMA_VERSION,
        "datasets": json.dumps(datasets),
        "built_at": datetime.now(timezone.utc).isoformat(),
        "checksum": checksum,
        "rule_count": len(rules),
        "attribute_count": 0,
    }
    con = sqlite3.connect(dest)
    try:
        con.execute("CREATE TABLE interaction_rules (id INTEGER PRIMARY KEY, source TEXT, payload TEXT)")
        con.execute("CREATE TABLE drug_attributes (ingredient TEXT PRIMARY KEY, payload TEXT)")
        con.execute("CREATE TABLE bundle_meta (key TEXT PRIMARY KEY, value TEXT)")
        con.executemany("INSERT INTO interaction_rules (source, payload) VALUES ('ddinter', ?)",
                        [(json.dumps(r),) for r in rules])
        con.executemany("INSERT INTO bundle_meta (key, value) VALUES (?, ?)", list(meta.items()))
        con.commit()
    finally:
        con.close()
    return meta
print('builder ready')


## 3. Download DDInter 2.0 DDI CSVs (per ATC category)

In [ ]:
!pip -q install pandas requests
import pandas as pd, requests, io
BASE = 'https://ddinter2.scbdd.com/static/media/download/ddinter_downloads_code_'
CATEGORIES = ['A', 'B', 'D', 'H', 'L', 'P', 'R', 'V']
frames = []
for c in CATEGORIES:
    r = requests.get(f'{BASE}{c}.csv', timeout=120); r.raise_for_status()
    frames.append(pd.read_csv(io.StringIO(r.text)))
    print(f'  {c}: {len(frames[-1])} rows')
raw = pd.concat(frames, ignore_index=True)
print('total:', len(raw)); raw.head()


## 4. Build + write the bundle

In [ ]:
def to_rows(df):
    for _, r in df.iterrows():
        yield RawInteraction(str(r.get('Drug_A', '')), str(r.get('Drug_B', '')),
                             str(r.get('Level', '')), str(r.get('Mechanism', '') or ''))
rules = build_rules(to_rows(raw))
meta = write_bundle(rules, 'bundle.sqlite', {'ddinter': '2.0'})
print(meta)


## 5. Download `bundle.sqlite`, then install it in *Admin → Interaction Bundle*

In [ ]:
from google.colab import files
files.download('bundle.sqlite')
